In [2]:
## Download the analysis result files produced by the workflow genomic in ABRomics

# Definition of the API (swagger): https://analysis.abromics.fr/api/schema/swagger

In [1]:
## imports
from dotenv import load_dotenv # to read the environment variables from the .env file
import os
import requests
import json
import pandas as pd

from rdflib import Graph
import uuid
from rdflib import Namespace
from rdflib import FOAF, DCTERMS, SKOS, VANN, TIME, RDF, RDFS, XSD, OWL, Literal

In [2]:
## env variables
load_dotenv() #! when the .env values change there is no reload it takes the old value... 

## load the environment variables
api_url = os.getenv('API_URL') #! not the right value..
api_username = os.getenv('API_USERNAME')
api_password = os.getenv('API_PASSWORD')
api_temp_basic_token = os.getenv('API_TEMP_BASIC_TOKEN') # will be changed at one point find a way to get a new token via a login with creds

In [3]:
## Objective: Getting all the file results for by analysis for each analysis that worked (aka status:ready_to_report)

# Finding the ids of all the analysis that worked
# Loading the file associated with one analysis that worked

################# Issues : the endpoint doesn't load correctly from the .env file
# 
# endpoint = f"{api_url}/api/analysis/"
# print(endpoint)
# print(api_temp_bearer_token)
# 

In [3]:
response = requests.get(
    "https://analysis.abromics.fr/api/analysis/",
    params={'status': 'ready_to_report'},
    headers = {
        'Authorization': f"Basic {api_temp_basic_token}",
    }
)
exploitable_analysis = response.json()
exploitable_analysis_ids = [analysis["id"] for analysis in exploitable_analysis]
exploitable_analysis_ids

[47, 54, 48, 42, 27, 50, 29, 43, 51, 44, 28, 45, 55, 71, 81, 91]

In [7]:
# Finding the report related to one exploitable analysis (ex: analysis with id 40)

In [4]:
#! the bearer token will expire at one point
response = requests.get(
    "https://analysis.abromics.fr/api/report/",
    params={'id': 40},
    headers = {
        'Authorization': f"Basic {api_temp_basic_token}",
    }
)
reports = response.json()

# save the reports in a local file
with open('reports.json', 'w') as f:
    json.dump(reports, f)

In [5]:
reports_names = [report["title"] for report in reports]
print(reports_names) ## Looks like duplicated reports bundle in one report from the database (data are different so no duplication)
print(len(reports_names)) ## 73 Genomic report => maybe one for each strain of bacteria found ?

['GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT', 'GENOMIC RESULT REPORT',

In [6]:
[print(section["title"]) for section in reports[0]["sections"]]

SAMPLE SUMMARY
ANALYSIS SUMMARY
 ANTIMICROBIAL RESISTANCE ANALYSES
ORGANISM IDENTIFICATION
QUALITY METRICS
QUALITY ASSEMBLY


[None, None, None, None, None, None]

In [7]:
[print(data) for data in reports[0]["sections"][0]["data"]] ## corespond to the metadata of the sample

{'type': 'summary', 'header': ['Sample ID', 'Microorganism scientific name', 'Collected at', 'Sample source', 'Sample type', 'Host', 'Location', 'Sequencing Technology', 'Sequencing Platform', 'Submitter name', 'Submitter email'], 'values': [None, 'Achromobacter xylosoxidans', None, 'Other', 'environmental', 'Other', None, '454 GS 20', '454', 'Admin Abromics', 'admin@analysis.abromics.fr']}


[None]

In [8]:
[print(data) for data in reports[0]["sections"][1]["data"]] ## corespond to "ANALYSIS SUMMARY"

{'type': 'summary', 'header': ['Isolate identified as', 'ST', 'Number of acquired genes resistances with known target antibiotics'], 'values': ['Pseudomonas aeruginosa', 235, 10]}
{'type': 'list', 'title': 'List of target antibiotics', 'values': [['streptomycin', 'gentamicin', 'kanamycin', 'ampicillin', 'ampicillin,amoxicillin/clavulanicacid,cefoxitin,ceftriaxone', 'chloramphenicol', 'ciprofloxacinI/R', 'fosfomycin', 'unknown[qacE_1_X68232]', 'sulfisoxazole']]}


[None, None]

In [9]:
[print(data) for data in reports[0]["sections"][2]["data"]] ## corespond to "ANTIMICROBIAL RESISTANCE ANALYSIS"

{'type': 'table', 'title': 'Acquired resistance genes', 'header': ['Resistance gene', 'Gene length', 'Identity %', 'Coverage %', 'Contig', 'Start in contig', 'End in contig', 'Strand', 'Target antibiotic', 'Accession number'], 'values': [['aadA13', "ant(2'')-Ia", "aph(3')-IIb", 'blaOXA-488', 'blaPAO', 'catB7', 'crpP', 'fosA', 'qacE', 'sul1'], [798, 534, 807, 789, 1194, 639, 198, 408, 278, 840], [100.0, 100.0, 98.76, 99.87, 99.25, 98.75, 98.48, 99.02, 100.0, 100.0], [100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 83.48, 100.0], ['contig00086', 'contig00102', 'contig00001', 'contig00039', 'contig00001', 'contig00033', 'contig00056', 'contig00009', 'contig00139', 'contig00080'], [133, 133, 169523, 19349, 182685, 25811, 3829, 183999, 1, 733], [930, 666, 170329, 20137, 183878, 26449, 4026, 184406, 278, 1572], ['-', '-', '+', '+', '-', '+', '+', '+', '-', '-'], ['streptomycin', 'gentamicin', 'kanamycin', 'ampicillin', 'ampicillin,amoxicillin/clavulanicacid,cefoxitin,ceftriaxone', 'c

[None, None, None]

In [10]:
## Create a structured dataframe with the results of the analysis
# df = pd.DataFrame.from_dict(reports[0]["sections"][2]["data"][0])

df = pd.DataFrame(reports[0]["sections"][2]["data"][0]["values"], index=reports[0]["sections"][2]["data"][0]["header"])
df

,0,1,2,3,4,5,6,7,8,9
Resistance gene,aadA13,ant(2'')-Ia,aph(3')-IIb,blaOXA-488,blaPAO,catB7,crpP,fosA,qacE,sul1
Gene length,798,534,807,789,1194,639,198,408,278,840
Identity %,100.0,100.0,98.76,99.87,99.25,98.75,98.48,99.02,100.0,100.0
Coverage %,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,83.48,100.0
Contig,contig00086,contig00102,contig00001,contig00039,contig00001,contig00033,contig00056,contig00009,contig00139,contig00080
Start in contig,133,133,169523,19349,182685,25811,3829,183999,1,733
End in contig,930,666,170329,20137,183878,26449,4026,184406,278,1572
Strand,-,-,+,+,-,+,+,+,-,-
Target antibiotic,streptomycin,gentamicin,kanamycin,ampicillin,"ampicillin,amoxicillin/clavulanicacid,cefoxiti...",chloramphenicol,ciprofloxacinI/R,fosfomycin,unknown[qacE_1_X68232],sulfisoxazole
Accession number,AY713504,X04555,CP006832,CP017969,AY083592,AF036933,HM560971,ACWU01000146,X68232,U12338


In [13]:
## Understand what is in the report, normally there is only one analysis but many ANTIMICROBIAL RESISTANCE ANALYSIS are
## available. There is maybe multiple strain of bacterias in the analysed sample.

In [11]:
## Checking out the ORGANISM IDENTIFICATION category find out if there is multiple species of bacterias in the sample 
## 26 different bacterias present in the sample

df = pd.DataFrame(reports[0]["sections"][3]["data"][0]["values"], index=reports[0]["sections"][3]["data"][0]["header"])
df

,0,1,2,3,4,5,6,7,8,9,...,69,70,71,72,73,74,75,76,77,78
Species name,Pseudomonas aeruginosa,Pseudomonas sp. 13159349,Pseudomonas putida,Pseudomonas stutzeri,Pseudomonas protegens,Pseudomonas phage MD8,Pseudomonas sp. PS1(2021),Pseudomonas sp. FDAARGOS 761,Stenotrophomonas maltophilia,Pseudomonas phage JBD30,...,Pseudomonas campi,Pseudomonas sp. p1(2021b),Pseudomonas sp. CC6-YY-74,Bordetella genomosp. 6,Cutibacterium acnes,Pseudomonas sp. ABC1,Pseudomonas balearica,Runella slithyformis,Pseudomonas fulva,Pseudomonas synxantha
NCBI ID,287,2662034,303,316,380021,1868596,2866282,2545800,40324,1223260,...,2731681,2874628,1930532,463024,1747,2748080,74829,106,47880,47883
Fraction of reads,0.98642,0.00328,0.00282,0.00196,0.00102,0.00059,0.00053,0.00044,0.0004,0.00036,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
## Create a small handmade ttl graph using the previous antibiotic resistance data + the metadata uploaded to ABRomics
## (which is at the start of the report category 0)
## Graph with only one analysis 

In [13]:
# Placeholder nmespaces
ABROMICS = Namespace("http://www.abromics.org/ontology/")
SOSA = Namespace('http://www.w3.org/ns/sosa/')
NCIt = Namespace('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl/')
OHMI = Namespace('https://bioportal.bioontology.org/ontologies/OHMI/')
ARO = Namespace('https://bioportal.bioontology.org/ontologies/ARO/')
GO = Namespace('https://bioportal.bioontology.org/ontologies/GO/')


g = Graph()

## Sample metadata
sampleMetadata = reports[0]["sections"][0]["data"][0]["values"]
sampleMetadata

## Storing the sample
sampleId = uuid.uuid1()
g.add((ABROMICS[str(sampleId)], RDF.type, SOSA.ObservableProperty))
g.add((ABROMICS[str(sampleId)], ABROMICS.MicroOrganism, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][1]).replace(' ', '_')]))
g.add((ABROMICS[str(sampleId)], ABROMICS.CollectedAt, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][2])]))
g.add((ABROMICS[str(sampleId)], ABROMICS.Source, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][3])]))
g.add((ABROMICS[str(sampleId)], ABROMICS.SampleType, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][4])]))
g.add((ABROMICS[str(sampleId)], ABROMICS.Host, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][5]).replace(' ', '_')]))
g.add((ABROMICS[str(sampleId)], ABROMICS.Location, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][6])]))
g.add((ABROMICS[str(sampleId)], ABROMICS.SequencingTechnology, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][7]).replace(' ', '_')]))
g.add((ABROMICS[str(sampleId)], ABROMICS.SequencingPlatform, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][8]).replace(' ', '_')]))
g.add((ABROMICS[str(sampleId)], ABROMICS.SubmitterName, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][9]).replace(' ', '_')]))
g.add((ABROMICS[str(sampleId)], ABROMICS.SubmitterEmail, ABROMICS[str(reports[0]["sections"][0]["data"][0]["values"][10]).replace(' ', '_')]))

## Storing the analysis
analysisId = uuid.uuid1()
g.add((ABROMICS[str(analysisId)], RDF.type, SOSA.Observation))
g.add((ABROMICS[str(analysisId)], SOSA.ObservedProperty, ABROMICS[str(sampleId)]))

## Storing the FeatureOfInterest
featureOfInterestId = uuid.uuid1()
g.add((ABROMICS[str(featureOfInterestId)], SOSA.isFeatureOfInterest, ABROMICS[str(analysisId)]))
g.add((ABROMICS[str(featureOfInterestId)], RDF.type, SOSA.FeatureOfInterest))
g.add((ABROMICS[str(featureOfInterestId)], RDF.type, ABROMICS.Genomic))

## Storing the Microbiome
microbiomeId = uuid.uuid1()
g.add((ABROMICS[str(microbiomeId)], RDF.type, OHMI.HumanMicrobiome))
g.add((ABROMICS[str(featureOfInterestId)], ABROMICS.has, ABROMICS[str(microbiomeId)]))

## Store a single strain observed in the sample
strainId = uuid.uuid1()
g.add((ABROMICS[str(strainId)], RDF.type, ARO.Strain))
g.add((ABROMICS[str(microbiomeId)], ABROMICS.contains, ABROMICS[str(strainId)]))
g.add((ABROMICS[str(strainId)], ABROMICS.MLSTProfile, Literal("MLST_profile", datatype=XSD.string)))
g.add((ABROMICS[str(strainId)], ABROMICS.STProfile, Literal("ST_profile", datatype=XSD.string)))

## Store a molecule the strain is resistant to
moleculeId = uuid.uuid1()
g.add((ABROMICS[str(moleculeId)], RDF.type, ARO.Molecule))
g.add((ABROMICS[str(strainId)], ABROMICS.isResistantTo, ABROMICS[str(moleculeId)]))
g.add((ABROMICS[str(moleculeId)], RDFS.label, Literal(str(reports[0]["sections"][2]["data"][0]["values"][8][0]), datatype=XSD.string))) ## store the name of the molecule


## Store the resistance gene associated with the molecule the bacteria is resistant to
geneId = uuid.uuid1()
g.add((ABROMICS[str(geneId)], RDF.type, GO.Gene))
g.add((ABROMICS[str(geneId)], ABROMICS.givesResistanceTo, ABROMICS[str(moleculeId)]))


g.serialize(destination="../graph/antibiotic-resistance.ttl",format='ttl')

<Graph identifier=N3e2de8348516491ab18d9f1e22a6bc4d (<class 'rdflib.graph.Graph'>)>

In [ ]:
## Issues of the graph: 
## A microbiome can't be an observed property as the sample is already one, it makes the graph confusing. A new abromics relation should be created to link 
## a FeatureOfInterest to the OHMI:Microbiome class

In [18]:
## Use their sparql endpoint https://www.bgee.org/support/tutorial-query-bgee-knowledge-graph-sparql (intergate their data to the graph)
## Maybe also the NCBI one too
## + UBERON